# 01 · Открытая глобальная карта полей Fields of The World (FTW), 2-е издание

**Что это.** Готовые полигоны полей, которые команда FTW посчитала нейросетью по Sentinel-2 для всего мира
за **2017–2025 гг.** (≈1,24 млрд полигонов, лицензия **CC-BY-4.0**), и растры вероятностей «поле»/«граница»
с шагом 2,5 м. Ничего обучать и запускать на GPU не нужно — данные просто вырезаются по вашему участку.
Источник: <https://source.coop/ftw/global-data-2e>, интерактивная карта:
<https://research.taylorgeospatial.org/global-ftw-2e/web/>.

**Что делает ноутбук.**
1. Вырезает полигоны FTW по вашему участку за выбранные годы → GeoPackage на Диске (слой на каждый год).
2. (по желанию) Пересчитывает вероятности «поле»/«граница» на сетку вашего композита → GeoTIFF.
   Этот GeoTIFF подаётся в конвейер v7 как дополнительные барьеры (гибрид: на Донбассе +0,035 IoU к v6).
3. (по желанию) Сравнивает FTW с вашей разметкой теми же метриками, что и конвейер, и рисует картинки.

**Важно про смысл данных.** Поле FTW — «поле в смысле ДЗЗ» (связная область, похожая на пашню), а не кадастр.
Модель обучена на 24 странах без России; пар/залежь в обучающей разметке отсутствовали, поэтому
залежь FTW часто пропускает. Результат на Донбассе (тест, 301 эталонное поле):

| слой | IoU | IoU≥0.7 | bF (границы) | охват эталона |
|---|---|---|---|---|
| FTW 2025 | 0,34 | 0,21 | 0,45 | 0,52 |
| FTW 2022 (лучший год) | 0,50 | 0,34 | 0,70 | 0,81 |
| ваш конвейер v6 | 0,64 | 0,55 | 0,71 | 0,93 |
| v7: v6 + карта границ FTW (`L_ext`) | 0,67 | 0,59 | 0,72 | 0,92 |
| **v7: + границы FTW/PRUE 2026 (ноутбук 02) + Higra** | **0,684** | **0,61** | 0,72 | 0,92 |

**Время:** вырезка полигонов — секунды на небольшой участок, минуты на область; GPU не нужен
(Среда выполнения → Сменить среду → CPU).

### Подключение Google Диска
Выполните ячейку и разрешите доступ к Диску (всплывёт окно Google). Все результаты пишутся в папку
`Colab_Notebooks/Agro_terra/research/…` — её можно поменять в параметрах ниже.

In [ ]:
import os, sys, json, time, glob, shutil, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
LOCAL_TEST = bool(os.environ.get('LOCAL_TEST'))          # True только при проверке вне Colab
if not LOCAL_TEST:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
else:
    DRIVE_ROOT = Path(os.environ['LOCAL_DRIVE'])
print('Диск:', DRIVE_ROOT, '| файлов в корне:', len(list(DRIVE_ROOT.iterdir())))

## Шаг 1. Параметры — заполните под себя
Участок задаётся **одним** из способов (используется первый заполненный):
- `COMPOSITE_PATH` — ваш композит (охват берётся из растра; на его сетку пишутся вероятности);
- `AOI_PATH` — граница участка (geojson/gpkg/shp);
- `BBOX` — рамка `(долгота_мин, широта_мин, долгота_макс, широта_макс)`.

Пути — относительно корня «Мой диск». Годы — от 2017 до 2025 (2026 появится, когда FTW его выпустит).

In [ ]:
COMPOSITE_PATH = 'Colab_Notebooks/Agro_terra/TAT/TA_composite_2026_0504_0910_8b.tif'
AOI_PATH = None                     # например 'Colab_Notebooks/Agro_terra/TAT/TAT.geojson'
BBOX = None                         # например (38.12, 47.76, 38.34, 47.93)
YEARS = [2022, 2023, 2024, 2025]    # за какие годы вырезать полигоны
MIN_SCORE = 0                       # отсечь полигоны с уверенностью модели ниже (0–100); 0 — оставить все

SAVE_PROBABILITY = True             # вероятности «поле»/«граница» на сетку композита (нужен COMPOSITE_PATH)
PROB_YEARS = [2022, 2023, 2025]     # по каким годам собирать границы (берётся максимум по годам)
PROB_BLOCK = 4096                   # размер блока в пикселях при записи большого растра

GT_PATH = None                      # ваша разметка для сравнения, например 'Colab_Notebooks/Agro_terra/test/gt_pieces.gpkg'
GT_TYPE_COL = 'type'                # колонка типа использования в разметке (если есть)

OUT_DIR = 'Colab_Notebooks/Agro_terra/research/ftw_global/out'
NAME = 'TAT'                        # префикс имён выходных файлов

## Шаг 2. Установка библиотек
В Colab почти всё уже есть; ячейка доустанавливает недостающее (≈30 с). Перезапуск среды не нужен.

In [ ]:
!pip -q install pyarrow pyogrio folium mapclassify > /dev/null
import numpy as np, pandas as pd, geopandas as gpd, shapely, rasterio
import pyarrow.parquet as pq, pyarrow.fs as pfs
from rasterio.warp import reproject, Resampling, transform_bounds
from rasterio.windows import Window
OUT = DRIVE_ROOT / OUT_DIR; OUT.mkdir(parents=True, exist_ok=True)
print('Результаты будут в', OUT)

## Шаг 3. Функции чтения FTW
Выполните ячейку без изменений. Файлы FTW лежат в открытом хранилище Source Cooperative (AWS, us-west-2) и
читаются кусками: из многогигабайтного файла зоны UTM подтягиваются только группы строк, попадающие в участок.

In [ ]:
FTW_BUCKET = 'us-west-2.opendata.source.coop/ftw/global-data-2e'
FTW_HTTP = 'https://data.source.coop/ftw/global-data-2e'
_fs = pfs.S3FileSystem(anonymous=True, region='us-west-2')


def utm_zones(bbox):
    x0, _, x1, _ = bbox
    return sorted({int((x + 180) // 6) + 1 for x in np.linspace(x0, x1, 16)})


def ftw_fields(bbox, year, min_score=0):
    """Полигоны FTW в рамке bbox (lon/lat) за год → GeoDataFrame EPSG:4326."""
    x0, y0, x1, y1 = bbox; parts = []
    for z in utm_zones(bbox):
        f = pq.ParquetFile(f'{FTW_BUCKET}/vector/{year}/zone={z:02d}/utm{z:02d}.parquet', filesystem=_fs)
        md = f.metadata; ix = {md.schema.column(i).path: i for i in range(md.num_columns)}
        sel = []
        for g in range(md.num_row_groups):
            rg = md.row_group(g); st = {k: rg.column(ix[f'bbox.{k}']).statistics for k in ('xmin', 'ymin', 'xmax', 'ymax')}
            if st['xmin'].min <= x1 and st['xmax'].max >= x0 and st['ymin'].min <= y1 and st['ymax'].max >= y0:
                sel.append(g)
        if not sel:
            continue
        df = f.read_row_groups(sel).to_pandas(); bb = df.pop('bbox')
        m = np.array([b['xmin'] <= x1 and b['xmax'] >= x0 and b['ymin'] <= y1 and b['ymax'] >= y0 for b in bb])
        df = df[m]
        parts.append(gpd.GeoDataFrame(df.drop(columns='geometry'), geometry=shapely.from_wkb(df.geometry.values), crs=4326))
    if not parts:
        return gpd.GeoDataFrame({'geometry': []}, geometry='geometry', crs=4326)
    g = gpd.GeoDataFrame(pd.concat(parts, ignore_index=True), crs=4326)
    g = g[g.intersects(shapely.box(*bbox)) & (g['score'] >= min_score)].reset_index(drop=True)
    g['area_ha'] = g['metrics:area'] / 1e4; g['year'] = year
    g['determination:datetime'] = g['determination:datetime'].astype(str)
    return g


_RIDX = None
def ftw_tiles(bbox, year):
    global _RIDX
    if _RIDX is None:
        _RIDX = pq.read_table(f'{FTW_BUCKET}/index/raster.parquet', filesystem=_fs,
                              columns=['year', 'tile_key', 'href', 'xmin', 'ymin', 'xmax', 'ymax']).to_pandas()
    t = _RIDX; x0, y0, x1, y1 = bbox
    return t[(t.year == year) & (t.xmin <= x1) & (t.xmax >= x0) & (t.ymin <= y1) & (t.ymax >= y0)]


GDAL_ENV = dict(GDAL_DISABLE_READDIR_ON_OPEN='EMPTY_DIR', CPL_VSIL_CURL_ALLOWED_EXTENSIONS='.tif',
                GDAL_HTTP_MULTIRANGE='YES', GDAL_HTTP_MERGE_CONSECUTIVE_RANGES='YES', GDAL_HTTP_MAX_RETRY='4')


def ftw_probability(bbox, year, dst_transform, dst_shape, dst_crs):
    """Вероятности «поле» (среднее) и «граница» (максимум) FTW на заданной сетке, 0–1; NaN — нет данных."""
    field = np.full(dst_shape, np.nan, np.float32); bnd = np.full(dst_shape, np.nan, np.float32)
    for _, r in ftw_tiles(bbox, year).iterrows():
        with rasterio.Env(**GDAL_ENV), rasterio.open('/vsicurl/' + r.href.replace(FTW_HTTP, FTW_BASE_URL)) as s:
            for band, out, rs in ((1, field, Resampling.average), (2, bnd, Resampling.max)):
                tmp = np.full(dst_shape, np.nan, np.float32)
                reproject(rasterio.band(s, band), tmp, src_nodata=None, dst_nodata=np.nan,
                          dst_transform=dst_transform, dst_crs=dst_crs, resampling=rs)
                ok = ~np.isnan(tmp); out[ok] = np.where(np.isnan(out[ok]), tmp[ok], np.fmax(out[ok], tmp[ok]))
    return field / 255.0, bnd / 255.0


FTW_BASE_URL = FTW_HTTP          # если data.source.coop недоступен: 'https://s3.us-west-2.amazonaws.com/us-west-2.opendata.source.coop/ftw/global-data-2e'
print('ok')

## Шаг 4. Определяем участок

In [ ]:
if COMPOSITE_PATH:
    with rasterio.open(DRIVE_ROOT / COMPOSITE_PATH) as s:
        GRID = dict(transform=s.transform, shape=(s.height, s.width), crs=s.crs)
        AOI_BBOX = transform_bounds(s.crs, 'EPSG:4326', *s.bounds)
    aoi_geom = None
elif AOI_PATH:
    aoi = gpd.read_file(DRIVE_ROOT / AOI_PATH).to_crs(4326); aoi_geom = aoi.union_all()
    AOI_BBOX = tuple(aoi.total_bounds); GRID = None
else:
    AOI_BBOX = tuple(BBOX); aoi_geom = None; GRID = None
w_km = (AOI_BBOX[2] - AOI_BBOX[0]) * 111 * np.cos(np.radians((AOI_BBOX[1] + AOI_BBOX[3]) / 2)); h_km = (AOI_BBOX[3] - AOI_BBOX[1]) * 111
print(f'Рамка участка: {np.round(AOI_BBOX, 4)} ≈ {w_km:.0f} × {h_km:.0f} км; зоны UTM: {utm_zones(AOI_BBOX)}')

## Шаг 5. Вырезаем полигоны FTW за выбранные годы
Для каждой зоны UTM читается только нужная часть файла. Результат — `<NAME>_ftw_fields.gpkg`, по слою на год
(`y2022`, `y2023`, …). Колонки: `score` — средняя уверенность модели внутри полигона (0–100; это рейтинг,
а не вероятность), `area_ha` — площадь, `year`.

In [ ]:
FTW = {}
gpkg = OUT / f'{NAME}_ftw_fields.gpkg'
for y in YEARS:
    t0 = time.time(); g = ftw_fields(AOI_BBOX, y, MIN_SCORE)
    if aoi_geom is not None:
        g = g[g.intersects(aoi_geom)].reset_index(drop=True)
    FTW[y] = g
    g.to_file(gpkg, layer=f'y{y}', driver='GPKG')
    print(f'{y}: {len(g):>7} полигонов, {g.area_ha.sum():>10.0f} га, медиана {g.area_ha.median() if len(g) else 0:.1f} га  ({time.time()-t0:.0f} с)')
print('Сохранено:', gpkg)

## Шаг 6 (по желанию). Вероятности «поле»/«граница» на сетку вашего композита
Пишется GeoTIFF `<NAME>_ftw_prob.tif` той же сетки, что композит (uint8, 0–255):
- канал 1 — **граница**: максимум вероятности границы по годам `PROB_YEARS`;
- канал 2 — **устойчивость границы**: в скольких годах вероятность границы ≥ 0,5;
- канал 3 — **поле**: средняя вероятность «поле» по годам.

Этот файл указывается в конвейере v7 в параметре `EXT_BOUNDARY` — тогда границы FTW станут
дополнительными барьерами при слиянии суперпикселей. Большой композит пишется блоками, это может занять
10–40 минут на область (работа идёт по сети, GPU не нужен).

In [ ]:
if SAVE_PROBABILITY and GRID is not None:
    out_tif = OUT / f'{NAME}_ftw_prob.tif'
    prof = dict(driver='GTiff', width=GRID['shape'][1], height=GRID['shape'][0], count=3, dtype='uint8',
                crs=GRID['crs'], transform=GRID['transform'], compress='deflate', tiled=True, blockxsize=512,
                blockysize=512, BIGTIFF='IF_SAFER', nodata=0)
    H, W = GRID['shape']; nb = ((H + PROB_BLOCK - 1) // PROB_BLOCK) * ((W + PROB_BLOCK - 1) // PROB_BLOCK); k = 0
    with rasterio.open(out_tif, 'w', **prof) as dst:
        for r0 in range(0, H, PROB_BLOCK):
            for c0 in range(0, W, PROB_BLOCK):
                win = Window(c0, r0, min(PROB_BLOCK, W - c0), min(PROB_BLOCK, H - r0)); k += 1
                wt = rasterio.windows.transform(win, GRID['transform']); shp = (int(win.height), int(win.width))
                wb = transform_bounds(GRID['crs'], 'EPSG:4326', *rasterio.windows.bounds(win, GRID['transform']))
                bmax = np.zeros(shp, np.float32); bcnt = np.zeros(shp, np.uint8); fsum = np.zeros(shp, np.float32); fn = 0
                for y in PROB_YEARS:
                    f, b = ftw_probability(wb, y, wt, shp, GRID['crs'])
                    if np.isnan(f).all():
                        continue
                    bmax = np.fmax(bmax, np.nan_to_num(b)); bcnt += (np.nan_to_num(b) >= .5).astype(np.uint8)
                    fsum += np.nan_to_num(f); fn += 1
                out = np.stack([np.clip(bmax * 254 + 1, 1, 255), bcnt + 1, np.clip(fsum / max(fn, 1) * 254 + 1, 1, 255)]).astype(np.uint8)
                dst.write(out, window=win)
                print(f'\rблок {k}/{nb}', end='')
    print('\nСохранено:', out_tif, '(значения 1–255; 0 — нет данных)')
else:
    print('Пропущено (нет композита или SAVE_PROBABILITY=False)')

### Функции оценки и картинок (общие для всех ноутбуков)
Ничего менять не нужно — просто выполните ячейку. Метрики те же, что в тестах конвейера
(`tests/eval_gt.py` в репозитории), поэтому числа можно сравнивать с v6:
- **IoU** — для каждого эталонного поля (≥ 5 га) берётся лучший по перекрытию полигон, среднее по полям;
- **IoU≥0.7** — доля эталонных полей, найденных «хорошо»;
- **frag** — сколько полигонов в среднем покрывают одно эталонное поле (1 — идеально, >1 — дробление);
- **merged** — сколько полигонов «склеили» 2 и больше эталонных полей;
- **bF** — точность границ (F-мера, допуск 2 пикселя), **P/R** — точность/полнота линий;
- **охват** — доля площади эталона, покрытая полигонами; по классам (если в эталоне есть колонка `type`).

In [ ]:
import numpy as np, geopandas as gpd, shapely, rasterio, rasterio.windows
from rasterio.features import rasterize
from rasterio.transform import from_origin
from scipy import ndimage as ndi


def load_gt(path, bbox=None):
    """Эталонная разметка: только полигоны в рамке bbox (EPSG:4326: lon_min, lat_min, lon_max, lat_max), без пустых
    и повторных. Так можно указывать разметку на всю область (например DNR_type_2026.geojson), а сравнивать на участке."""
    box = gpd.GeoSeries([shapely.box(*bbox)], crs=4326) if bbox is not None else None
    g = gpd.read_file(path, bbox=box)
    g = g[g.geometry.notna()]
    if bbox is not None:
        g = g[g.to_crs(4326).representative_point().within(box.iloc[0]).values]
    g = g[~g.geometry.to_wkb().duplicated()].reset_index(drop=True)
    print('эталон:', len(g), 'полигонов', '(в рамке участка)' if bbox is not None else '')
    return g


def grid_for(gdf, res=10.0):
    """Сетка 10 м в UTM по охвату слоя (если нет вашего композита)."""
    utm = gdf.estimate_utm_crs()
    x0, y0, x1, y1 = gdf.to_crs(utm).total_bounds
    w, h = int(np.ceil((x1 - x0) / res)) + 2, int(np.ceil((y1 - y0) / res)) + 2
    return from_origin(x0 - res, y1 + res, res, res), (h, w), utm


def evaluate(pred, gt, transform=None, shape=None, crs=None, min_ha=5, type_col='type', label=''):
    """pred, gt — GeoDataFrame полигонов (любая СК). transform/shape/crs — сетка для растровых метрик
    (по умолчанию 10 м UTM по эталону). Печатает строку метрик и возвращает dict."""
    if transform is None:
        transform, shape, crs = grid_for(gt)
    H, W = shape
    gt = gt[gt.geometry.notna()].copy(); pred = pred[pred.geometry.notna()].copy()
    gt = gt[~gt.geometry.to_wkb().duplicated()]                # в выгрузках разметки бывают повторы одного полигона
    box = shapely.box(*rasterio.transform.array_bounds(H, W, transform))
    gt = gt[gt.to_crs(crs).representative_point().within(box).values]   # эталон — только в пределах сетки (участка)
    if not len(gt):
        print(label, ': в пределах участка нет эталонных полей'); return dict(label=label, gt=0)
    win = rasterio.windows.from_bounds(*gt.to_crs(crs).total_bounds, transform)          # сетка — только охват эталона
    win = win.round_offsets().round_lengths()
    c0, r0 = max(int(win.col_off) - 20, 0), max(int(win.row_off) - 20, 0)
    W, H = min(int(win.col_off + win.width) + 20, W) - c0, min(int(win.row_off + win.height) + 20, H) - r0
    transform = rasterio.windows.transform(rasterio.windows.Window(c0, r0, W, H), transform)
    if H * W > 4e8:
        print(f'ВНИМАНИЕ: эталон охватывает {H}×{W} пикселей — оценка займёт много памяти; задайте участок поменьше (BBOX)')
    gt['geometry'] = shapely.make_valid(gt.geometry.values); pred['geometry'] = shapely.make_valid(pred.geometry.values)
    ea = 6933                                                  # равновеликая СК для площадей
    g6 = gt.to_crs(ea); g6 = g6[g6.area / 1e4 >= min_ha].reset_index(drop=True)
    p6 = pred.to_crs(ea); p6['geometry'] = shapely.make_valid(p6.geometry.values)
    tree = shapely.STRtree(p6.geometry.values)
    ious, frag, merged = [], [], np.zeros(len(p6), int)
    for g in g6.geometry.values:
        best, nfr = 0.0, 0
        for j in tree.query(g, predicate='intersects'):
            pj = p6.geometry.values[j]; inter = g.intersection(pj).area
            if inter >= .15 * g.area: merged[j] += 1
            if inter >= .05 * g.area: nfr += 1
            best = max(best, inter / g.union(pj).area)
        ious.append(best); frag.append(nfr)
    ious = np.array(ious)
    idmap = lambda d: rasterize([(g, i + 1) for i, g in enumerate(d.to_crs(crs).geometry.values)],
                                out_shape=(H, W), transform=transform, fill=0, dtype='int32')
    GT, P = idmap(g6), idmap(p6)
    def bnd(m):
        b = np.zeros(m.shape, bool); b[:, :-1] |= m[:, :-1] != m[:, 1:]; b[:-1] |= m[:-1] != m[1:]; return b
    gb, pb = bnd(GT) & (GT > 0), bnd(P) & (P > 0)
    dgt, dpb = ndi.distance_transform_edt(~gb), ndi.distance_transform_edt(~pb)
    dom = ndi.binary_dilation(GT > 0, iterations=3)
    prec = (dgt[pb & dom] <= 2).mean() if (pb & dom).any() else 0.0
    rec = (dpb[gb] <= 2).mean() if gb.any() else 0.0
    bF = 2 * prec * rec / (prec + rec + 1e-9)
    cov = ((P > 0) & (GT > 0)).sum() / max((GT > 0).sum(), 1)
    res = dict(label=label, gt=len(g6), pred=len(pred), IoU=round(float(ious.mean()), 3) if len(ious) else 0,
               IoU07=round(float(np.mean(ious >= .7)), 2) if len(ious) else 0, frag=round(float(np.mean(frag)), 2),
               merged=int((merged >= 2).sum()), bF=round(float(bF), 3), P=round(float(prec), 2), R=round(float(rec), 2),
               cover=round(float(cov), 2))
    if type_col in gt.columns:
        gtt = gt.to_crs(crs)
        for t in sorted(gtt[type_col].dropna().unique()):
            m = rasterize([(g, 1) for g in gtt[gtt[type_col] == t].geometry.values], out_shape=(H, W),
                          transform=transform, fill=0, dtype='uint8') > 0
            res[f'cover_{t}'] = round(float((P[m] > 0).mean()), 2) if m.any() else None
    print(' | '.join(f'{k} {v}' for k, v in res.items()))
    return res


def show_overlay(img_path, layers, title='', bbox=None, out_png=None, bands=(1, 2, 3), figsize=(11, 11)):
    """Картинка: подложка — ваш растр (3 канала, растяжка 2–98 %), поверх контуры слоёв.
    layers — список (GeoDataFrame, цвет, подпись)."""
    import matplotlib.pyplot as plt
    from rasterio.windows import from_bounds
    with rasterio.open(img_path) as s:
        win = from_bounds(*bbox, s.transform) if bbox is not None else rasterio.windows.Window(0, 0, s.width, s.height)
        k = max(1.0, max(win.width, win.height) / 2500)                  # большие охваты читаются с прореживанием
        shp = (len(bands), int(win.height / k), int(win.width / k))
        a = s.read(list(bands), window=win, boundless=True, out_shape=shp).astype(float)
        t = s.window_transform(win) * rasterio.Affine.scale(win.width / shp[2], win.height / shp[1]); crs = s.crs
    for i in range(a.shape[0]):
        lo, hi = np.nanpercentile(a[i][a[i] > 0], [2, 98]) if (a[i] > 0).any() else (0, 1)
        a[i] = np.clip((a[i] - lo) / (hi - lo + 1e-9), 0, 1)
    ext = (t.c, t.c + t.a * a.shape[2], t.f + t.e * a.shape[1], t.f)
    fig, ax = plt.subplots(figsize=figsize)
    ax.imshow(np.moveaxis(a, 0, -1), extent=ext)
    for gdf, color, name in layers:
        if gdf is not None and len(gdf):
            gdf.to_crs(crs).boundary.plot(ax=ax, color=color, linewidth=.8, label=name)
    ax.set_xlim(ext[0], ext[1]); ax.set_ylim(ext[2], ext[3]); ax.set_title(title); ax.legend(loc='upper right')
    if out_png:
        fig.savefig(out_png, dpi=130, bbox_inches='tight')
    plt.show()

## Шаг 7 (по желанию). Сравнение с вашей разметкой
Нужен `GT_PATH`. Если указан композит — метрики считаются на его сетке, иначе на сетке 10 м по разметке.

In [ ]:
if GT_PATH:
    gt = load_gt(DRIVE_ROOT / GT_PATH, AOI_BBOX)
    gt_bbox = tuple(gt.to_crs(4326).total_bounds)
    rows = []
    for y, g in FTW.items():
        gg = g[g.intersects(shapely.box(*gt_bbox))]
        kw = dict(transform=GRID['transform'], shape=GRID['shape'], crs=GRID['crs']) if GRID else {}
        rows.append(evaluate(gg, gt, type_col=GT_TYPE_COL, label=f'FTW {y}', **kw))
    pd.DataFrame(rows).to_csv(OUT / f'{NAME}_ftw_vs_gt.csv', index=False)
    display(pd.DataFrame(rows))
else:
    print('GT_PATH не задан — сравнение пропущено')

## Шаг 8. Картинка: композит + полигоны FTW (+ эталон)
Показывается фрагмент `VIEW_BBOX` (по умолчанию — охват разметки или центр участка 10×10 км).

In [ ]:
VIEW_BBOX = None          # (lon_min, lat_min, lon_max, lat_max) или None
if VIEW_BBOX is None:
    gb = tuple(gt.to_crs(4326).total_bounds) if GT_PATH and len(gt) else None
    if gb and gb[2] - gb[0] < 0.4 and gb[3] - gb[1] < 0.3:        # разметка на небольшой участок — показываем её охват
        VIEW_BBOX = gb
    else:
        cx, cy = (AOI_BBOX[0] + AOI_BBOX[2]) / 2, (AOI_BBOX[1] + AOI_BBOX[3]) / 2
        VIEW_BBOX = (cx - .07, cy - .045, cx + .07, cy + .045)
if COMPOSITE_PATH:
    vb = transform_bounds('EPSG:4326', GRID['crs'], *VIEW_BBOX)
    colors = ['cyan', 'white', 'lime', 'orange', 'yellow']
    lays = [(g[g.intersects(shapely.box(*VIEW_BBOX))], colors[i % 5], f'FTW {y}') for i, (y, g) in enumerate(FTW.items())][-2:]
    if GT_PATH:
        lays.append((gt[gt.to_crs(4326).intersects(shapely.box(*VIEW_BBOX))], 'red', 'эталон'))
    show_overlay(DRIVE_ROOT / COMPOSITE_PATH, lays, title=f'{NAME}: FTW', bbox=vb, out_png=OUT / f'{NAME}_ftw_view.png')

## Шаг 9 (по желанию). Интерактивная карта поверх спутниковой подложки
Показывает последний год из `YEARS` в окне `VIEW_BBOX` (не более 20 000 полигонов, чтобы не тормозило).

In [ ]:
import folium
y = YEARS[-1]; g = FTW[y]; g = g[g.intersects(shapely.box(*VIEW_BBOX))].head(20000)
m = folium.Map(location=[(VIEW_BBOX[1] + VIEW_BBOX[3]) / 2, (VIEW_BBOX[0] + VIEW_BBOX[2]) / 2], zoom_start=13,
               tiles='https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}', attr='Esri')
folium.GeoJson(g[['score', 'area_ha', 'geometry']], name=f'FTW {y}',
               style_function=lambda f: {'color': 'yellow', 'weight': 1, 'fillOpacity': 0},
               tooltip=folium.GeoJsonTooltip(['score', 'area_ha'])).add_to(m)
if GT_PATH:
    folium.GeoJson(gt.to_crs(4326)[lambda d: d.intersects(shapely.box(*VIEW_BBOX))], name='эталон',
                   style_function=lambda f: {'color': 'red', 'weight': 1, 'fillOpacity': 0}).add_to(m)
folium.LayerControl().add_to(m)
m

## Что дальше
- Посмотрите, в какие годы FTW лучше видит ваши поля (шаг 7): на Донбассе лучшим оказался 2022, а 2024–2025 —
  заметно хуже (много пропусков на залежи и части посевов).
- Файл `<NAME>_ftw_prob.tif` из шага 6 подключается к конвейеру v7 (`EXT_BOUNDARY`, пресет `L_ext`),
  см. ИНСТРУКЦИЮ в папке `research`.
- Полигоны FTW можно использовать как готовый слой «полей» в QGIS или как псевдо-разметку для дообучения
  (ноутбук 04) — но только после проверки глазами.